# Step 4: MCP 통합 — 외부 도구 생태계 연결

## 학습 목표

- **MCP(Model Context Protocol)**가 에이전트의 도구 확장성을 어떻게 제공하는지 이해합니다
- **stdio 전송 기반 JSON-RPC** 프로토콜의 메시지 흐름을 이해합니다
- 빌트인 도구와 MCP 도구가 **동일한 Tool 인터페이스**로 통합되는 패턴을 배웁니다

> **MCP란?**
> Model Context Protocol은 LLM 애플리케이션이 외부 도구 서버와 통신하기 위한 표준 프로토콜입니다.
> 파일시스템, 데이터베이스, API 등 다양한 서버를 동일한 프로토콜로 연결할 수 있어,
> 에이전트의 도구 생태계를 무한히 확장할 수 있습니다.

---

## Claude Code 소스 분석

### 4-1. MCP 아키텍처 전체 그림

Claude Code에서 MCP는 세 가지 레이어로 구성됩니다:

```
┌─────────────────────────────────────────────────────────┐
│  에이전트 루프 (queryLoop)                                │
│    └─ assembleToolPool()  ← 빌트인 + MCP 도구를 합침     │
│         ├─ 빌트인 도구: Bash, Read, Write, ...           │
│         └─ MCP 도구: MCPTool(server__toolName)           │
├─────────────────────────────────────────────────────────┤
│  MCPTool  (src/tools/MCPTool/MCPTool.tsx)               │
│    └─ Tool 인터페이스를 구현하여 래핑                      │
├─────────────────────────────────────────────────────────┤
│  MCPClient  (src/services/mcp/mcpClient.ts)             │
│    └─ JSON-RPC over stdio/SSE/HTTP/WebSocket            │
├─────────────────────────────────────────────────────────┤
│  MCP 서버 (외부 프로세스)                                 │
│    예: filesystem, database, github, slack, ...          │
└─────────────────────────────────────────────────────────┘
```

핵심 통찰: **에이전트 루프는 도구가 빌트인인지 MCP인지 전혀 알 필요가 없습니다.** 모든 도구가 동일한 `Tool` 인터페이스를 구현하기 때문입니다.

### 4-2. MCPClient — 4가지 전송 타입 (mcpClient.ts)

```typescript
// src/services/mcp/mcpClient.ts — MCP 클라이언트 핵심 구조

// 4가지 전송(transport) 타입 지원
type MCPTransport = "stdio" | "sse" | "http" | "websocket";

// 연결 라이프사이클:
//   1. connect()    → 서버 프로세스 시작 (stdio) 또는 HTTP 연결
//   2. initialize() → 프로토콜 버전 협상 (JSON-RPC handshake)
//   3. listTools()  → 사용 가능한 도구 목록 획득
//   4. callTool()   → 도구 실행 요청/응답

// stdio transport의 경우:
//   - 서버를 subprocess로 시작
//   - stdin으로 JSON-RPC 요청을 보내고
//   - stdout에서 JSON-RPC 응답을 읽음
//   - 줄 단위(newline-delimited) JSON 사용
```

**stdio가 가장 일반적인 이유:**
- 서버를 로컬 프로세스로 실행하므로 네트워크 설정이 필요 없음
- 보안: 로컬 프로세스 간 통신이므로 인증 불필요
- Claude Code의 대부분의 MCP 서버가 stdio를 사용

### 4-3. MCPTool — MCP 도구를 Tool로 래핑 (MCPTool.tsx)

```typescript
// src/tools/MCPTool/MCPTool.tsx — MCP 도구를 Tool 인터페이스로 래핑

// MCPTool은 일반 Tool과 동일한 인터페이스를 구현합니다:
//   - name:        "serverName__toolName" (네임스페이스로 충돌 방지)
//   - description: MCP 서버가 제공한 도구 설명
//   - inputSchema: MCP 서버가 제공한 JSON Schema
//   - call():      MCPClient.callTool()을 호출

// 네이밍 규칙: "서버이름__도구이름"
// 예: "filesystem__read_file", "github__create_issue"
// 이중 언더스코어(__)로 서버와 도구를 구분합니다.
```

### 4-4. assembleToolPool() — 빌트인 + MCP 통합 (tools.ts:345+)

```typescript
// src/tools.ts:345+ — assembleToolPool()

function assembleToolPool(mcpServers, builtinTools): Tool[] {
  // 1. 빌트인 도구 수집
  const tools: Tool[] = [...builtinTools];

  // 2. 각 MCP 서버에서 도구 목록을 가져와 MCPTool로 래핑
  for (const server of mcpServers) {
    const mcpTools = server.listTools();
    for (const t of mcpTools) {
      tools.push(new MCPTool(server.name, t));
    }
  }

  // 3. 결과: [Bash, Read, Write, ..., filesystem__read, github__create_issue, ...]
  return tools;
}

// 에이전트 루프에서는 이 통합된 배열을 그대로 사용:
// for (const toolCall of response.toolCalls) {
//   const tool = tools.find(t => t.name === toolCall.name);
//   const result = await tool.call(toolCall.input);  // 빌트인이든 MCP든 동일
// }
```

---

## Python으로 구현하기

`mini_claude/mcp/client.py`에 구현된 MCP 클라이언트를 확인합니다.

### 핵심 클래스 구조

1. **MCPServerConfig** — 서버 연결 설정 (command, args, env)
2. **MCPClient** — stdio 기반 JSON-RPC 통신 (connect, call_tool, disconnect)
3. **MCPTool** — MCP 도구를 Tool 인터페이스로 래핑 (Adapter 패턴)

In [ ]:
import sys, os
sys.path.insert(0, ".")

from mini_claude.mcp.client import MCPServerConfig, MCPClient, MCPTool
from mini_claude.tool_base import Tool, ToolResult

# --- MCPServerConfig: 서버 연결 설정 ---
# Claude Code의 settings.json에서 MCP 서버를 선언하는 방식과 동일합니다.
config = MCPServerConfig(
    name="filesystem",
    command="npx",
    args=["-y", "@modelcontextprotocol/server-filesystem", "/tmp"],
    env={},  # 추가 환경변수 (현재 환경에 병합됨)
)

print("MCPServerConfig 구조:")
print(f"  name:    {config.name}")
print(f"  command: {config.command}")
print(f"  args:    {config.args}")
print()

# --- MCPTool이 Tool 인터페이스를 구현하는지 확인 ---
print(f"MCPTool은 Tool의 서브클래스인가? {issubclass(MCPTool, Tool)}")
print()

# MCPTool의 네이밍 규칙 시연 (mock client으로)
class MockClient:
    async def call_tool(self, name, args):
        return ToolResult(data=f"Mock result for {name}")

mock_tool = MCPTool(
    server_name="filesystem",
    tool_name="read_file",
    tool_description="Read the contents of a file",
    tool_input_schema={
        "type": "object",
        "properties": {"path": {"type": "string"}},
        "required": ["path"],
    },
    client=MockClient(),
)

print(f"MCPTool 인스턴스:")
print(f"  name:         {mock_tool.name}")          # "filesystem__read_file"
print(f"  description:  {mock_tool.description}")
print(f"  input_schema: {mock_tool.input_schema}")
print(f"  is_read_only: {mock_tool.is_read_only()}")  # False (fail-closed)

### JSON-RPC 메시지 흐름 시뮬레이션

MCP 서버를 실제로 실행하지 않고도, JSON-RPC 프로토콜의 메시지 흐름을 확인할 수 있습니다.
아래 코드는 `MCPClient`가 서버와 주고받는 메시지를 시뮬레이션합니다.

In [ ]:
import json
from mini_claude.mcp.client import _make_request

# --- Step 1: initialize 핸드셰이크 ---
print("=" * 60)
print("MCP 연결 과정 (JSON-RPC 메시지 흐름)")
print("=" * 60)

# 클라이언트 → 서버: initialize 요청
init_request = _make_request("initialize", {
    "protocolVersion": "2024-11-05",
    "capabilities": {},
    "clientInfo": {"name": "mini-claude", "version": "0.1.0"},
})
print("\n[1] Client -> Server: initialize")
print(json.dumps(json.loads(init_request), indent=2, ensure_ascii=False))

# 서버 → 클라이언트: initialize 응답 (시뮬레이션)
init_response = {
    "jsonrpc": "2.0",
    "id": 1,
    "result": {
        "protocolVersion": "2024-11-05",
        "capabilities": {"tools": {}},
        "serverInfo": {"name": "filesystem-server", "version": "1.0.0"},
    },
}
print("\n[2] Server -> Client: initialize response")
print(json.dumps(init_response, indent=2, ensure_ascii=False))

# 클라이언트 → 서버: initialized 알림 (id 없음 = notification)
initialized_notification = {
    "jsonrpc": "2.0",
    "method": "notifications/initialized",
}
print("\n[3] Client -> Server: initialized (notification, no id)")
print(json.dumps(initialized_notification, indent=2, ensure_ascii=False))

# --- Step 2: tools/list ---
tools_request = _make_request("tools/list", {})
print("\n[4] Client -> Server: tools/list")
print(json.dumps(json.loads(tools_request), indent=2, ensure_ascii=False))

tools_response = {
    "jsonrpc": "2.0",
    "id": 2,
    "result": {
        "tools": [
            {
                "name": "read_file",
                "description": "Read the complete contents of a file",
                "inputSchema": {
                    "type": "object",
                    "properties": {"path": {"type": "string"}},
                    "required": ["path"],
                },
            },
            {
                "name": "write_file",
                "description": "Write content to a file",
                "inputSchema": {
                    "type": "object",
                    "properties": {
                        "path": {"type": "string"},
                        "content": {"type": "string"},
                    },
                    "required": ["path", "content"],
                },
            },
        ]
    },
}
print("\n[5] Server -> Client: tools/list response")
print(json.dumps(tools_response, indent=2, ensure_ascii=False))

# --- Step 3: tools/call ---
call_request = _make_request("tools/call", {
    "name": "read_file",
    "arguments": {"path": "/tmp/hello.txt"},
})
print("\n[6] Client -> Server: tools/call")
print(json.dumps(json.loads(call_request), indent=2, ensure_ascii=False))

call_response = {
    "jsonrpc": "2.0",
    "id": 3,
    "result": {
        "content": [
            {"type": "text", "text": "Hello, World!"}
        ],
        "isError": False,
    },
}
print("\n[7] Server -> Client: tools/call response")
print(json.dumps(call_response, indent=2, ensure_ascii=False))

### 실습: 빌트인 도구 + MCP 도구 통합

빌트인 도구(Bash, Read)와 MCP 도구(mock)를 하나의 도구 풀에 합쳐서,
에이전트 루프가 구분 없이 사용할 수 있는지 확인합니다.
이것이 Claude Code의 `assembleToolPool()` 패턴입니다.

In [ ]:
import asyncio
from mini_claude.tool_base import Tool, ToolResult
from mini_claude.tool_registry import ToolRegistry
from mini_claude.tools.bash_tool import BashTool
from mini_claude.tools.file_read_tool import FileReadTool
from mini_claude.mcp.client import MCPTool

# --- 1. 빌트인 도구 등록 ---
registry = ToolRegistry()
registry.register(BashTool)
registry.register(FileReadTool)

# --- 2. Mock MCP 도구 생성 ---
# 실제로는 MCPClient.connect()가 서버에서 도구 목록을 받아 MCPTool을 생성합니다.
# 여기서는 mock으로 시뮬레이션합니다.

class MockMCPClient:
    """MCP 서버를 시뮬레이션하는 mock 클라이언트"""
    async def call_tool(self, tool_name, arguments):
        if tool_name == "query":
            return ToolResult(data=f"[Mock DB] SELECT result for: {arguments}")
        elif tool_name == "get_issues":
            return ToolResult(data=f"[Mock GitHub] Issues: #1 Bug fix, #2 Feature request")
        return ToolResult(error=f"Unknown tool: {tool_name}")

mock_client = MockMCPClient()

# MCP 도구들을 MCPTool로 래핑
mcp_tools = [
    MCPTool(
        server_name="database",
        tool_name="query",
        tool_description="Execute a SQL query on the database",
        tool_input_schema={
            "type": "object",
            "properties": {"sql": {"type": "string", "description": "SQL query"}},
            "required": ["sql"],
        },
        client=mock_client,
    ),
    MCPTool(
        server_name="github",
        tool_name="get_issues",
        tool_description="List GitHub issues for a repository",
        tool_input_schema={
            "type": "object",
            "properties": {"repo": {"type": "string"}},
            "required": ["repo"],
        },
        client=mock_client,
    ),
]

# --- 3. assembleToolPool(): 빌트인 + MCP 통합 ---
for mcp_tool in mcp_tools:
    registry.register(mcp_tool)

# --- 4. 통합된 도구 풀 확인 ---
print("통합된 도구 풀 (assembleToolPool 결과):")
print("-" * 50)
for tool in registry.get_all_tools():
    tool_type = "MCP" if isinstance(tool, MCPTool) else "빌트인"
    print(f"  [{tool_type:4s}] {tool.name:30s} - {tool.description[:50]}")

# --- 5. 에이전트 루프처럼 도구를 이름으로 찾아 실행 ---
print("\n에이전트 루프 시뮬레이션:")
print("-" * 50)

test_calls = [
    ("Bash", {"command": "echo 'hello'"}),
    ("database__query", {"sql": "SELECT * FROM users LIMIT 5"}),
    ("github__get_issues", {"repo": "anthropics/claude-code"}),
]

for tool_name, args in test_calls:
    tool = registry.find_tool(tool_name)
    if tool:
        result = await tool.call(args)
        print(f"  {tool_name}: {result.content[:60]}")
    else:
        print(f"  {tool_name}: [도구를 찾을 수 없음]")

---

### 실습: MCPClient의 전체 연결 라이프사이클

MCPClient의 `connect()` 메서드가 수행하는 5단계를 코드로 확인합니다.
실제 MCP 서버 없이도 각 단계에서 어떤 JSON-RPC 메시지가 오가는지 이해할 수 있습니다.

In [ ]:
import json

# MCPClient.connect()의 5단계를 시뮬레이션
print("MCPClient.connect() 라이프사이클")
print("=" * 60)

steps = [
    ("1. subprocess 시작",
     "서버 프로세스를 시작합니다 (stdio transport)",
     "asyncio.create_subprocess_exec(config.command, *config.args, stdin=PIPE, stdout=PIPE)"),

    ("2. initialize 핸드셰이크",
     "프로토콜 버전을 협상합니다",
     json.dumps({"jsonrpc": "2.0", "id": 1, "method": "initialize",
                 "params": {"protocolVersion": "2024-11-05",
                           "clientInfo": {"name": "mini-claude"}}}, indent=4)),

    ("3. initialized 알림",
     "핸드셰이크 완료를 알립니다 (notification: id 없음)",
     json.dumps({"jsonrpc": "2.0", "method": "notifications/initialized"}, indent=4)),

    ("4. tools/list 요청",
     "사용 가능한 도구 목록을 요청합니다",
     json.dumps({"jsonrpc": "2.0", "id": 2, "method": "tools/list", "params": {}}, indent=4)),

    ("5. MCPTool 래핑",
     "각 도구를 MCPTool(Tool 인터페이스)로 래핑하여 반환합니다",
     'return [MCPTool(server_name, t["name"], t["description"], t["inputSchema"], self) for t in tools]'),
]

for title, desc, code in steps:
    print(f"\n--- {title} ---")
    print(f"  설명: {desc}")
    print(f"  코드/메시지:")
    for line in code.split("\n"):
        print(f"    {line}")

print("\n" + "=" * 60)
print("connect() 후에는 MCPTool 리스트가 반환되어 ToolRegistry에 등록됩니다.")
print("이후 call_tool()로 도구를 실행하고, disconnect()로 서버를 종료합니다.")

---

## 핵심 설계 원칙 정리

### 1. 표준 프로토콜로 무한 확장
MCP는 JSON-RPC 기반의 표준 프로토콜입니다. 어떤 언어, 어떤 플랫폼이든 MCP 서버를 구현하면 Claude Code의 도구로 사용할 수 있습니다. 이것이 Claude Code가 파일시스템, 데이터베이스, GitHub, Slack 등 수백 개의 외부 도구를 지원하는 방법입니다.

### 2. Adapter 패턴으로 인터페이스 통합
`MCPTool`은 MCP 프로토콜의 도구를 `Tool` 인터페이스로 래핑하는 Adapter입니다. 에이전트 루프는 `tool.call(args)`만 호출하면 되고, 그 도구가 로컬 빌트인인지 외부 MCP 서버인지 알 필요가 없습니다.

### 3. 네임스페이스로 충돌 방지
`"server__tool"` 형식의 네이밍 규칙으로 서로 다른 MCP 서버의 동일한 이름 도구가 충돌하지 않습니다. 예를 들어 `filesystem__read_file`과 `s3__read_file`은 다른 도구입니다.

### 4. Fail-closed 보안
`MCPTool.is_read_only()`는 항상 `False`를 반환합니다. 외부 서버의 도구가 읽기 전용인지 확신할 수 없으므로, 안전한 쪽(쓰기 가능)으로 가정합니다. 이것이 Claude Code의 "fail-closed" 철학입니다.

---

## 다음 Step 미리보기

**Step 5: 컨텍스트 관리**에서는 에이전트가 LLM에 보내는 **시스템 프롬프트**를 어떻게 조립하는지 살펴봅니다:
- **정적/동적 경계**로 프롬프트 캐시를 최적화하는 방법
- `@lru_cache`(memoize)로 반복 계산을 피하는 방법
- `ToolUseContext`가 도구 실행 환경을 캡슐화하는 패턴